## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

#groq_api_key=os.getenv("GROQ_API_KEY")
#groq_api_key



True

In [2]:
# from langchain_groq import ChatGroq
# model=ChatGroq(model="Gemma2-9b-It",groq_api_key=groq_api_key)
from langchain_ollama import ChatOllama
model = ChatOllama(model="phi4")
model

ChatOllama(model='phi4')

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")])

AIMessage(content="Hello, Krish! It's great to meet you. As a Chief AI Engineer, you're likely involved in some fascinating work with artificial intelligence technologies. If there's anything specific you'd like to discuss about your role, challenges you're facing, or advancements in AI that interest you, feel free to share. I'm here to help with any questions or topics related to AI and machine learning!", additional_kwargs={}, response_metadata={'model': 'phi4', 'created_at': '2026-02-12T19:33:46.5953947Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24224400900, 'load_duration': 175575900, 'prompt_eval_count': 23, 'prompt_eval_duration': 2252733600, 'eval_count': 80, 'eval_duration': 21693856900, 'logprobs': None, 'model_name': 'phi4'}, id='run--019c5358-0125-7851-be37-f17c3c415320-0', usage_metadata={'input_tokens': 23, 'output_tokens': 80, 'total_tokens': 103})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer"),
        AIMessage(content="Hello Krish! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Krish, and you are a Chief AI Engineer. Is there anything specific you'd like to discuss or explore further related to your role or projects? Let me know how I can assist!", additional_kwargs={}, response_metadata={'model': 'phi4', 'created_at': '2026-02-12T19:33:58.5252998Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11912560600, 'load_duration': 121621400, 'prompt_eval_count': 94, 'prompt_eval_duration': 1302381700, 'eval_count': 41, 'eval_duration': 10361698300, 'logprobs': None, 'model_name': 'phi4'}, id='run--019c5358-5ff3-75f3-8fd6-09527a1e1952-0', usage_metadata={'input_tokens': 94, 'output_tokens': 41, 'total_tokens': 135})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Krish and I am a Chief AI Engineer")],
    config=config
)

In [8]:
response.content

"Hello Krish! It's great to hear from you. As a Chief AI Engineer, you must be involved in some fascinating projects. If there's anything specific you'd like to discuss or need assistance with—whether it's about AI technologies, strategies for implementation, or any other related topics—I'm here to help. Feel free to share what's on your mind!"

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Krish. If you have any questions or need further assistance, feel free to ask!', additional_kwargs={}, response_metadata={'model': 'phi4', 'created_at': '2026-02-12T19:34:28.3998713Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7753695500, 'load_duration': 131633400, 'prompt_eval_count': 113, 'prompt_eval_duration': 1921579500, 'eval_count': 21, 'eval_duration': 5576917800, 'logprobs': None, 'model_name': 'phi4'}, id='run--019c5358-e4e3-7a82-b535-8f5ae90c42dc-0', usage_metadata={'input_tokens': 113, 'output_tokens': 21, 'total_tokens': 134})

In [10]:
## change the config-->session id
config2={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

"I'm sorry, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. You haven't provided that information, so I can't determine your name. If you'd like to share more context or details (within a safe and appropriate scope), feel free to do so!"

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config2
)
response.content

'Hello John! Nice to meet you. How can I assist you today?'

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config2
)
response.content

"Based on our recent conversation, your name is John. If there's anything specific you need help with or if you have any questions, feel free to ask!"

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [14]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")]})

AIMessage(content="Hello, Krish! How can I assist you today? If there's anything specific you'd like to know or discuss, feel free to let me know. 😊", additional_kwargs={}, response_metadata={'model': 'phi4', 'created_at': '2026-02-12T19:35:18.5912951Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10937846600, 'load_duration': 116623600, 'prompt_eval_count': 38, 'prompt_eval_duration': 1208215800, 'eval_count': 34, 'eval_duration': 9571119700, 'logprobs': None, 'model_name': 'phi4'}, id='run--019c5359-9abe-7872-ae0f-de378f707237-0', usage_metadata={'input_tokens': 38, 'output_tokens': 34, 'total_tokens': 72})

In [15]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [16]:
config = {"configurable": {"session_id": "chat4"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content="Hello Krish! How can I assist you today? If there's anything specific you'd like to know or discuss, feel free to let me know. 😊", additional_kwargs={}, response_metadata={'model': 'phi4', 'created_at': '2026-02-12T19:35:27.875339Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9227272900, 'load_duration': 100589600, 'prompt_eval_count': 38, 'prompt_eval_duration': 340541200, 'eval_count': 33, 'eval_duration': 8758000300, 'logprobs': None, 'model_name': 'phi4'}, id='run--019c5359-c776-7b52-9757-43aa0eee281b-0', usage_metadata={'input_tokens': 38, 'output_tokens': 33, 'total_tokens': 71})

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

"Your name is Krish, as you mentioned earlier. Is there anything else you'd like to talk about or any questions I can help with?"

In [18]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"French"})
response.content

"Bonjour ! Je m'appelle Assistant Virtuel. Comment puis-je vous aider aujourd'hui ? Si vous avez des questions ou besoin d'informations, n'hésitez pas à me demander. 😊"

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [21]:
config = {"configurable": {"session_id": "chat5"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"English"},
    config=config
)
repsonse.content

'Hello, Krish! How can I assist you today? If you have any questions or need information on something specific, feel free to let me know. 😊'

In [22]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [23]:
response.content

"Your name is Krish. If there's anything else you'd like to ask or discuss, feel free to tell me! 🌟"

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [24]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

ImportError: Could not import transformers python package. This is needed in order to calculate get_token_ids. Please install it with `pip install transformers`.

In [40]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"As an AI, I don't have access to your personal preferences like your favorite ice cream flavor.  \n\nWhat's your favorite ice cream? 😊🍦\n"

In [41]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked "whats 2 + 2" 😊  \n\n\n\n'

In [42]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [43]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I don't have access to past conversations or any personal information about you, including your name.  \n\nIf you'd like to tell me your name, I'd be happy to know! 😊  \n\n"

In [44]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"As a large language model, I have no memory of past conversations. If you'd like to ask me a math problem, I'm happy to help! 😊  Just let me know what it is. \n\n"